In [1]:

%matplotlib inline
import jax.numpy as jnp
from jax import value_and_grad
from jax import grad
from jax import random
import pandas as pd
from scipy.optimize import minimize
from scipy.stats import norm
import matplotlib.pyplot as plt
import seaborn as snb
import numpy as np
from scipy.stats import norm as norm_dist
from jax import value_and_grad
from jax import hessian

###Bayesian ML functions###

from bayesian_ml import *
from packages.BayesianLinearRegression import BayesianLinearRegression
from packages.LogisticRegression import LogisticRegression
from packages.Grid2D import Grid2D
from packages.LaplaceApproximation import LaplaceApproximation
from packages.PosteriorPredictiveDistribution import PosteriorPredictiveDistribution
from packages.Hyperparameters import Hyperparameters
from packages.StationaryIsotropicKernel import StationaryIsotropicKernel
from packages.GaussianProcessRegression import GaussianProcessRegression
from packages.BayesianLinearSoftmax import BayesianLinearSoftmax
from packages.metropolis import metropolis

from IPython.display import Markdown, display


###Distributions###
from scipy.stats import multivariate_normal as mvn
from scipy.stats import poisson

from packages.util_funs import sigmoid
from packages.util_funs import log_npdf

snb.set_theme(font_scale=1.25)
key = random.key(seed = 0)

In [2]:
def to_latex_matrix(matrix):
    """Convert a JAX/numpy matrix to a LaTeX bmatrix string."""
    rows = []
    for row in matrix:
        rows.append(" & ".join(f"{val:.2f}" for val in row))
    return r"\begin{bmatrix}" + r" \\ ".join(rows) + r"\end{bmatrix}"

def to_latex_vector(vector):
    """Convert a JAX/numpy vector to a LaTeX bmatrix column vector."""
    rows = r" \\ ".join(f"{val:.2f}" for val in vector)
    return r"\begin{bmatrix}" + rows + r"\end{bmatrix}"

# Part 1:

$$\prod^{N}_{n = 1} \mathcal{N}(y_{n}| f(\bf{x_{n}}| \bf{w}), \sigma^{2})$$

Which gives

$$\mathcal{N}(\bf{y} | \bf{\Phi w}, \sigma^{2} \bf{I})$$

Where $\Phi$ is given by 

In [15]:
def design_matrix(x):
    return jnp.column_stack((jnp.ones(len(x)), x, x**2, x**3))

Xtrain = jnp.array([0, 1, 2, 4, 5])
ytrain = jnp.array([1, 0.5, -0.1, -0.9, 1.1])

Phi = design_matrix(Xtrain)
display(Markdown(rf"$\Phi$ = ${to_latex_matrix(Phi)}$"))


$\Phi$ = $\begin{bmatrix}1.00 & 0.00 & 0.00 & 0.00 \\ 1.00 & 1.00 & 1.00 & 1.00 \\ 1.00 & 2.00 & 4.00 & 8.00 \\ 1.00 & 4.00 & 16.00 & 64.00 \\ 1.00 & 5.00 & 25.00 & 125.00\end{bmatrix}$

## Question 1.2

The MLE is given by

$$\hat{w}_{\text{MLE}} = (\Phi^{T} \Phi)^{-1} \Phi^{T} \bm{y}$$

In [16]:
def w_MLE(phi):
    return jnp.linalg.solve(phi.T@phi, phi.T@ytrain).ravel()

MLE_est = w_MLE(phi=Phi)

display(Markdown(rf"The MLE estimate is given by $\hat{{w}}_{{MLE}} = (\Phi^{{T}} \Phi)^{{-1}} \Phi^{{T}} \bm{{y}} = {to_latex_vector(MLE_est)}$"))

The MLE estimate is given by $\hat{w}_{MLE} = (\Phi^{T} \Phi)^{-1} \Phi^{T} \bm{y} = \begin{bmatrix}0.95 \\ 0.27 \\ -0.69 \\ 0.13\end{bmatrix}$

## Question 1.3

The posterior $P(w|y)$ is given by

$$\bf{m} = \beta \bf{S} \Phi^{T} \bf{y}$$
where
$$\bf{S} = (\alpha \bf{I} + \beta \Phi^{T} \Phi)^{-1}$$